# Nectar IoT Data Platform — walkthrough

Runs the whole submission end to end and shows the output of each task.

**Before running:** `make setup && make generate`.

Set `FORMAT = "delta"` when the Delta jars are reachable; `"parquet"` runs offline.

In [1]:
import json, os, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)

FORMAT = "parquet"          # "delta" in production

from nectar.config import load_config
from nectar.logging_utils import RunContext, setup_logging
from nectar.spark_session import get_spark
from nectar.io_layer import read_table, resolve_format

setup_logging("WARNING")
cfg = load_config()
cfg.data["storage"]["table_format"] = FORMAT
spark = get_spark(cfg, "notebook")
FMT = cfg.data["_resolved_format"] = resolve_format(spark, FORMAT)
print("spark", spark.version, "| storage format", FMT)

Picked up JAVA_TOOL_OPTIONS: -Djavax.net.ssl.trustStore=/root/.ccr/java-truststore.p12 -Djavax.net.ssl.trustStorePassword=changeit -Djavax.net.ssl.trustStoreType=PKCS12 -Dhttps.proxyHost=127.0.0.1 -Dhttps.proxyPort=36431 -Dhttp.nonProxyHosts=localhost|127.0.0.1|::1|127.*|0.*|::|169.254.*|anthropic.com|*.anthropic.com|*.anthropic.com|registry.npmjs.org|jsr.io|npm.jsr.io|pypi.org|files.pythonhosted.org|index.crates.io|proxy.golang.org|host.docker.internal|10.*|172.16.*|172.17.*|172.18.*|172.19.*|172.20.*|172.21.*|172.22.*|172.23.*|172.24.*|172.25.*|172.26.*|172.27.*|172.28.*|172.29.*|172.30.*|172.31.*|192.168.*|100.64.0.0/10|*.svc.cluster.local|*.svc.cluster.local -Djdk.http.auth.tunneling.disabledSchemes= -Djdk.http.auth.proxying.disabledSchemes=
Picked up JAVA_TOOL_OPTIONS: -Djavax.net.ssl.trustStore=/root/.ccr/java-truststore.p12 -Djavax.net.ssl.trustStorePassword=changeit -Djavax.net.ssl.trustStoreType=PKCS12 -Dhttps.proxyHost=127.0.0.1 -Dhttps.proxyPort=36431 -Dhttp.nonProxyHosts=lo

26/08/17 11:53:47 WARN Utils: Your hostname, vm resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/08/17 11:53:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/17 11:53:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


spark 3.5.3 | storage format parquet


## 1. The dataset

Synthetic, but built to be *checkable*: defects are injected at known rates and the
manifest records the counts, so the quality framework's output can be reconciled
against ground truth.

In [2]:
manifest = json.loads((cfg.layer_path("raw") / "_generation_manifest.json").read_text())
print(f"sites={manifest['sites']}  buildings={manifest['buildings']}  assets={manifest['assets']}")
print(f"telemetry rows={manifest['telemetry_rows']:,}  events={manifest['event_rows']:,}")
print(f"window {manifest['window_start'][:10]} .. {manifest['window_end'][:10]}")

print("\ndegraded assets (should dominate the fault query):")
for a in manifest["degraded_assets"]:
    print("  ", a)
print("\nsilent assets (should appear in the 24h-silence query):")
for a in manifest["silent_assets"]:
    print("  ", a)
print("\nsite-wide excursions (should be flagged by the anomaly query):")
for s, w in manifest["site_anomalies"].items():
    print(f"   {s}  from {w[0][:10]}")

sites=3  buildings=9  assets=85
telemetry rows=607,268  events=3,373
window 2026-07-27 .. 2026-08-17

degraded assets (should dominate the fault query):
   SITE-BLR-BLD-01-CHILL-008
   SITE-BLR-BLD-01-CHILL-010
   SITE-BLR-BLD-02-COMPR-005
   SITE-BLR-BLD-03-UPS-002
   SITE-CBE-BLD-02-PUMP-002
   SITE-SIN-BLD-01-BOILE-003

silent assets (should appear in the 24h-silence query):
   SITE-CBE-BLD-02-PRESS-009
   SITE-CBE-BLD-03-COMPR-006
   SITE-SIN-BLD-01-UPS-004
   SITE-SIN-BLD-02-FLOWS-006

site-wide excursions (should be flagged by the anomaly query):
   SITE-CBE  from 2026-08-14
   SITE-SIN  from 2026-08-14


## 2. Run the pipeline

bronze → silver → gold → hierarchy → quality report.
Skip this cell if `make pipeline` has already been run.

In [3]:
from nectar.pipeline.run_batch import run_pipeline

ctx = RunContext()
run_pipeline(cfg, ["bronze", "silver", "gold", "hierarchy", "report"], ctx=ctx)
print(json.dumps(ctx.metrics, indent=2))

26/08/17 11:53:50 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


26/08/17 11:54:06 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


26/08/17 11:54:07 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


{
  "stage.bronze.seconds": 15.85,
  "silver.assets.orphans": 6,
  "silver.telemetry.rows": 595885,
  "silver.events.rows": 3301,
  "stage.silver.seconds": 35.39,
  "gold.tables": 15,
  "stage.gold.seconds": 37.78,
  "hierarchy.closure_rows": 136,
  "hierarchy.orphans": 6,
  "hierarchy.disconnected": 18,
  "stage.hierarchy.seconds": 3.83,
  "quality.verdict": "PASS",
  "quality.pass_rate_pct": 100.0,
  "stage.quality_report.seconds": 2.84,
  "elapsed_seconds": 96.4
}


## 3. Data quality (Task 5)

Does the framework rediscover what the generator injected?

In [4]:
report = json.loads((cfg.layer_path("quality") / "reports" / "data_quality_report_latest.json").read_text())
t = report["totals"]
print(f"verdict {report['verdict']}   rules {t['rules_evaluated']}   breached {t['rules_breached']}")
print(f"rows {t['rows_evaluated']:,}   quarantined {t['rows_quarantined']:,}   pass rate {t['pass_rate_pct']}%")

print("\nrules that caught something:")
for r in report["rules"]:
    if r["rows_failed"]:
        print(f"  {r['rule_id']:<45} {r['severity']:<8} {r['rows_failed']:>6}  {r['failure_rate']*100:6.3f}%")

verdict PASS   rules 35   breached 0
rows 611,124   quarantined 11,938   pass rate 100.0%

rules that caught something:
  tel.uniqueness.business_key                   BLOCKING   7966   1.311%
  tel.timeliness.late_arrival                   WARN       7534   1.240%
  evt.uniqueness.event_id                       BLOCKING     34   1.007%
  evt.timeliness.late_arrival                   WARN         32   0.948%
  evt.completeness.message_present              WARN         28   0.830%
  evt.completeness.severity_not_null            BLOCKING     26   0.770%
  tel.completeness.operating_mode_present       WARN       2039   0.335%
  tel.completeness.vibration_present            WARN       2020   0.332%
  tel.completeness.power_consumption_present    WARN       2012   0.331%
  tel.completeness.humidity_present             WARN       1988   0.327%
  tel.completeness.temperature_present          WARN       1974   0.325%
  tel.completeness.pressure_present             WARN       1952   0.321%
  te

In [5]:
by_rule = {r["rule_id"]: r["rows_failed"] for r in report["rules"]}
ts_defects = by_rule["tel.validity.timestamp_parseable"] + by_rule["tel.validity.timestamp_plausible"]
print("unknown asset ids   :", by_rule["tel.consistency.asset_registered"])
print("timestamp defects   :", ts_defects)
print("out-of-range values :", sum(v for k, v in by_rule.items() if k.startswith("tel.accuracy")))

quarantined = read_table(spark, cfg.table_path("quarantine", "telemetry"), FMT)
print("\nquarantined rows keep every original column plus the rules they broke:")
quarantined.select("asset_id", "timestamp", "_quarantine_reasons").show(5, truncate=60)

unknown asset ids   : 1695
timestamp defects   : 1822
out-of-range values : 2428

quarantined rows keep every original column plus the rules they broke:
+-----------------------+-------------------+-----------------------------------------+
|               asset_id|          timestamp|                      _quarantine_reasons|
+-----------------------+-------------------+-----------------------------------------+
|SITE-BLR-BLD-01-AHU-004|2026-07-27 05:25:00|[tel.accuracy.power_consumption_in_range]|
|SITE-BLR-BLD-01-AHU-004|2026-07-29 07:55:00|            [tel.uniqueness.business_key]|
|SITE-BLR-BLD-01-AHU-004|2026-08-02 03:05:00|            [tel.uniqueness.business_key]|
|SITE-BLR-BLD-01-AHU-004|2026-08-02 14:40:00|      [tel.accuracy.temperature_in_range]|
|SITE-BLR-BLD-01-AHU-004|2026-08-03 19:55:00|            [tel.uniqueness.business_key]|
+-----------------------+-------------------+-----------------------------------------+
only showing top 5 rows



In [6]:
from nectar.quality.engine import freshness_report

tel = read_table(spark, cfg.table_path("silver", "telemetry"), FMT)
fresh = freshness_report(tel, cfg, ctx.batch_id)
print("devices that went dark (freshness is how absence is detected):")
(fresh.filter("is_stale")
      .select("asset_id", "site_id", "last_seen_at", "lag_minutes")
      .orderBy("lag_minutes", ascending=False)
      .show(10, truncate=False))

devices that went dark (freshness is how absence is detected):


+-------------------------+--------+-------------------+------------------+
|asset_id                 |site_id |last_seen_at       |lag_minutes       |
+-------------------------+--------+-------------------+------------------+
|SITE-SIN-BLD-02-FLOWS-006|SITE-SIN|2026-08-14 20:10:00|3825.4333333333334|
|SITE-SIN-BLD-01-UPS-004  |SITE-SIN|2026-08-15 13:20:00|2795.4333333333334|
|SITE-CBE-BLD-02-PRESS-009|SITE-CBE|2026-08-15 22:30:00|2245.4333333333334|
|SITE-CBE-BLD-03-COMPR-006|SITE-CBE|2026-08-16 07:15:00|1720.4333333333334|
+-------------------------+--------+-------------------+------------------+



## 4. Asset hierarchy (Task 4)

The closure table and the graph model must agree.

In [7]:
hierarchy = read_table(spark, cfg.table_path("gold", "dim_asset_hierarchy"), FMT)
closure = read_table(spark, cfg.table_path("gold", "asset_closure"), FMT)

print(f"closure rows: {closure.count()}   max depth: {hierarchy.agg({'level': 'max'}).first()[0]}")
hierarchy.groupBy("connectivity_status").count().orderBy("count", ascending=False).show()

closure rows: 136   max depth: 2
+-------------------+-----+
|connectivity_status|count|
+-------------------+-----+
|          CONNECTED|   67|
|         STANDALONE|   12|
|           ORPHANED|    6|
+-------------------+-----+



In [8]:
from nectar.hierarchy.closure_table import downstream_impacted, orphan_assets

focus = hierarchy.orderBy("descendant_count", ascending=False).first()["asset_id"]
print("blast radius of", focus)
(downstream_impacted(hierarchy, closure, focus)
    .select("asset_id", "asset_type", "depth", "path")
    .show(truncate=False))

print("orphans (parent pointer resolved to nothing):")
orphan_assets(hierarchy).select("asset_id", "site_id", "connectivity_status").show(truncate=False)

blast radius of SITE-CBE-BLD-01-CHILL-008


+-------------------------+-----------+-----+-------------------------------------------------------------------------------+
|asset_id                 |asset_type |depth|path                                                                           |
+-------------------------+-----------+-----+-------------------------------------------------------------------------------+
|SITE-CBE-BLD-01-AHU-009  |AHU        |1    |SITE-CBE-BLD-01-CHILL-008 > SITE-CBE-BLD-01-AHU-009                            |
|SITE-CBE-BLD-01-AHU-010  |AHU        |1    |SITE-CBE-BLD-01-CHILL-008 > SITE-CBE-BLD-01-AHU-010                            |
|SITE-CBE-BLD-01-AHU-013  |AHU        |1    |SITE-CBE-BLD-01-CHILL-008 > SITE-CBE-BLD-01-AHU-013                            |
|SITE-CBE-BLD-01-TEMPS-011|Temp Sensor|2    |SITE-CBE-BLD-01-CHILL-008 > SITE-CBE-BLD-01-AHU-010 > SITE-CBE-BLD-01-TEMPS-011|
|SITE-CBE-BLD-01-TEMPS-012|Temp Sensor|2    |SITE-CBE-BLD-01-CHILL-008 > SITE-CBE-BLD-01-AHU-010 > SITE-CBE-BLD-01-TEM

+-----------------------+--------+-------------------+
|asset_id               |site_id |connectivity_status|
+-----------------------+--------+-------------------+
|SITE-BLR-BLD-01-UPS-014|SITE-BLR|ORPHANED           |
|SITE-BLR-BLD-01-UPS-015|SITE-BLR|ORPHANED           |
|SITE-CBE-BLD-01-UPS-015|SITE-CBE|ORPHANED           |
|SITE-CBE-BLD-01-UPS-016|SITE-CBE|ORPHANED           |
|SITE-SIN-BLD-01-UPS-006|SITE-SIN|ORPHANED           |
|SITE-SIN-BLD-01-UPS-007|SITE-SIN|ORPHANED           |
+-----------------------+--------+-------------------+



In [9]:
from nectar.hierarchy import graph_model as gm

cols = ["asset_id", "asset_name", "asset_type", "site_id", "building_id", "parent_asset_id"]
assets = [r.asDict() for r in read_table(spark, cfg.table_path("silver", "assets"), FMT).select(*cols).collect()]
sites = [r.asDict() for r in read_table(spark, cfg.table_path("silver", "sites"), FMT)
         .select("site_id", "site_name").collect()]
buildings = [r.asDict() for r in read_table(spark, cfg.table_path("silver", "buildings"), FMT)
             .select("building_id", "building_name", "site_id").collect()]

g = gm.build_graph(assets, sites, buildings)
print(json.dumps(gm.topology_report(g), indent=2, default=str))

print("\nmost critical assets by blast radius:")
for aid, radius in gm.critical_assets(g, 5):
    print(f"  {aid:<34} {radius}")

sql_answer = {r["asset_id"]: r["depth"] for r in downstream_impacted(hierarchy, closure, focus).collect()}
print("\nclosure table and graph agree:", sql_answer == gm.downstream_impacted(g, focus))

{
  "asset_nodes": 85,
  "edges": 43,
  "roots": 42,
  "leaves": 56,
  "weakly_connected_components": 42,
  "largest_component_size": 7,
  "max_depth": 2,
  "is_forest": true,
  "multi_parent_assets": [],
  "cycles_detected": 0,
  "example_cycles": [],
  "isolated_assets": 0
}

most critical assets by blast radius:
  SITE-CBE-BLD-01-CHILL-008          6
  SITE-BLR-BLD-01-CHILL-003          4
  SITE-BLR-BLD-01-CHILL-010          3
  SITE-BLR-BLD-01-AHU-004            2
  SITE-BLR-BLD-01-AHU-011            2



closure table and graph agree: True


## 5. SQL challenge (Task 6)

Executed against the gold layer through DuckDB.

In [10]:
from nectar.serving.load_duckdb import build_database, connect
from nectar.serving.run_queries import run_analytics

build_database(cfg)
out = run_analytics(cfg)
for name, r in out["results"].items():
    print(f"  {name:<38} {r['status']}  {r.get('rows', 0):>6} rows")

  q1_top_energy_assets                   OK      10 rows
  q2_avg_daily_energy_per_site           OK       3 rows
  q3_assets_over_10_faults_30d           OK       6 rows
  q4_assets_silent_24h                   OK      10 rows
  q5_hourly_building_utilization         OK    4644 rows
  q6_site_power_anomalies                OK       4 rows


In [11]:
from nectar.serving.load_duckdb import connect
from nectar.serving.run_queries import _split_statements

con = connect(cfg)

def run_file(path):
    """DuckDB's execute() returns None for a script that opens with comments,
    so strip the comment header the same way the query runner does."""
    return con.execute(_split_statements(open(path).read())[0]).fetchdf()

print("Q1 - top energy consumers")
run_file("sql/analytics/q1_top_energy_assets.sql").head()

Q1 - top energy consumers


,rank,asset_id,asset_name,asset_type,manufacturer,site_id,building_id,rated_power_kw,total_energy_kwh,avg_daily_kwh,peak_power_kw,load_factor_pct,energy_share_pct
0,1,SITE-BLR-BLD-01-CHILL-008,Chiller-08,Chiller,Grundfos,SITE-BLR,SITE-BLR-BLD-01,320.0,71337.76,3242.63,530.63,43.3,7.00
1,2,SITE-SIN-BLD-02-CHILL-008,Chiller-08,Chiller,Atlas Copco,SITE-SIN,SITE-SIN-BLD-02,320.0,70052.89,3184.22,472.48,42.5,6.87
2,3,SITE-CBE-BLD-01-CHILL-008,Chiller-08,Chiller,Trane,SITE-CBE,SITE-CBE-BLD-01,320.0,69727.95,3169.45,523.48,42.3,6.84
3,4,SITE-BLR-BLD-01-CHILL-010,Chiller-10,Chiller,Trane,SITE-BLR,SITE-BLR-BLD-01,320.0,69394.38,3154.29,569.88,42.2,6.81
4,5,SITE-BLR-BLD-01-CHILL-003,Chiller-03,Chiller,Siemens,SITE-BLR,SITE-BLR-BLD-01,320.0,67824.93,3082.95,422.65,41.2,6.65


In [12]:
print("Q4 - assets silent for 24h (expect the generator's silent devices)")
run_file("sql/analytics/q4_assets_silent_24h.sql")

Q4 - assets silent for 24h (expect the generator's silent devices)


,asset_id,asset_name,asset_type,site_id,building_id,connectivity_status,last_reading_at,lifetime_readings,sensors_seen,hours_since_last_reading,status
0,SITE-BLR-BLD-01-UPS-014,UPS-14,UPS,SITE-BLR,SITE-BLR-BLD-01,ORPHANED,NaT,<NA>,<NA>,NaN,NEVER_REPORTED
1,SITE-BLR-BLD-01-UPS-015,UPS-15,UPS,SITE-BLR,SITE-BLR-BLD-01,ORPHANED,NaT,<NA>,<NA>,NaN,NEVER_REPORTED
2,SITE-CBE-BLD-01-UPS-015,UPS-15,UPS,SITE-CBE,SITE-CBE-BLD-01,ORPHANED,NaT,<NA>,<NA>,NaN,NEVER_REPORTED
3,SITE-CBE-BLD-01-UPS-016,UPS-16,UPS,SITE-CBE,SITE-CBE-BLD-01,ORPHANED,NaT,<NA>,<NA>,NaN,NEVER_REPORTED
4,SITE-SIN-BLD-01-UPS-006,UPS-06,UPS,SITE-SIN,SITE-SIN-BLD-01,ORPHANED,NaT,<NA>,<NA>,NaN,NEVER_REPORTED
5,SITE-SIN-BLD-01-UPS-007,UPS-07,UPS,SITE-SIN,SITE-SIN-BLD-01,ORPHANED,NaT,<NA>,<NA>,NaN,NEVER_REPORTED
6,SITE-SIN-BLD-02-FLOWS-006,Flow Sensor-06,Flow Sensor,SITE-SIN,SITE-SIN-BLD-02,CONNECTED,2026-08-14 20:10:00,5364,1,63.3,SILENT
7,SITE-SIN-BLD-01-UPS-004,UPS-04,UPS,SITE-SIN,SITE-SIN-BLD-01,STANDALONE,2026-08-15 13:20:00,5555,1,46.1,SILENT
8,SITE-CBE-BLD-02-PRESS-009,Pressure Sensor-09,Pressure Sensor,SITE-CBE,SITE-CBE-BLD-02,CONNECTED,2026-08-15 22:30:00,5666,1,36.9,SILENT
9,SITE-CBE-BLD-03-COMPR-006,Compressor-06,Compressor,SITE-CBE,SITE-CBE-BLD-03,CONNECTED,2026-08-16 07:15:00,5767,1,28.2,SILENT


In [13]:
print("Q6 - abnormal site power (expect the injected site excursions)")
result = run_file("sql/analytics/q6_site_power_anomalies.sql")
con.close()
result

Q6 - abnormal site power (expect the injected site excursions)


,site_id,site_name,city,event_date,energy_kwh,baseline_kwh,baseline_days,energy_zscore,pct_vs_baseline,pct_vs_same_weekday,peak_power_kw,anomaly_severity,detected_by
0,SITE-SIN,Changi Business Hub,Singapore,2026-08-14,26086.58,14804.02,7,1.47,76.2,31.9,472.48,MEDIUM,week_over_week
1,SITE-CBE,Nectar Coimbatore Campus,Coimbatore,2026-08-14,21703.68,12478.93,7,1.43,73.9,33.6,523.48,MEDIUM,week_over_week
2,SITE-CBE,Nectar Coimbatore Campus,Coimbatore,2026-08-15,4060.49,13259.24,7,-1.26,-69.4,34.1,118.69,MEDIUM,week_over_week
3,SITE-SIN,Changi Business Hub,Singapore,2026-08-15,4719.94,15705.37,7,-1.27,-69.9,29.9,123.66,MEDIUM,week_over_week


## 6. Real-time pipeline (Bonus A)

Same rules, same lakehouse, seconds instead of an hour. Runs against a file source,
so no broker is required.

In [14]:
from nectar.streaming.consumer import streaming_rules

print(f"{len(streaming_rules(cfg))} of the batch rules also run in the stream")
print("excluded: dedupe (handled by the watermark) and late-arrival (needs the landing partition)")
print("\nrun it with:  make stream        (or: python -m nectar.streaming.consumer --source file --once)")

20 of the batch rules also run in the stream
excluded: dedupe (handled by the watermark) and late-arrival (needs the landing partition)

run it with:  make stream        (or: python -m nectar.streaming.consumer --source file --once)


## 7. Wrap up

In [15]:
gold_tables = sorted(p.name for p in cfg.layer_path("gold").iterdir() if p.is_dir())
print(f"{len(gold_tables)} gold tables:")
for t in gold_tables:
    print("  ", t)
spark.stop()

18 gold tables:
   agg_asset_daily
   agg_building_daily
   agg_site_daily
   asset_closure
   curated_daily_asset_utilization
   curated_daily_environment
   curated_fault_statistics
   curated_hourly_energy
   dim_asset
   dim_asset_hierarchy
   dim_building
   dim_date
   dim_site
   dim_time
   fact_energy_hourly
   fact_event
   fact_telemetry
   stream_asset_5min
